# LLM-DM：冻结配置的 Colab 运行入口

先在 Colab 的「运行时 → 更改运行时类型」选择 GPU。默认只准备环境；训练单元有独立开关。

- 主实验：multi-scale → 1B / 19500；fixed-1B 从同一份可见数据取子集。
- K=128；先完整配置 seed 0，再正式 seeds 38–45。
- 训练的是优化方法的代理模型，不是重新预训练 1B LLM。
- 一次只跑一个方法 × 设置 × seed。数据与预算不在本 notebook 里重新切分或调整。
- GPU、内存和可用时长由 Colab 分配，不保证所有方法能在一次会话内完成。

代码、数据及配置来自下面固定的提交，不跟随 main 分支。


In [ ]:
from pathlib import Path
import json, re, subprocess, sys, os, platform, hashlib, importlib.metadata

RELEASE_REVISION = "2a40f564dd33d117d322a57746d43fc174180cdf"
UPSTREAM_REVISION = "37269969a0957448d51622e0c083977bc5d260e8"
REPO = Path("/content/llmdm_repo")
UPSTREAM = Path("/content/data-recipes")
if platform.system() != "Linux" or not Path("/content").is_dir():
    raise RuntimeError("请在 Google Colab 中运行此 notebook。")
if not re.fullmatch(r"[0-9a-f]{40}", RELEASE_REVISION):
    raise RuntimeError("发布版本未固定，禁止执行。")

def checkout_exact(url, destination, revision):
    if destination.exists():
        actual = subprocess.check_output(["git", "-C", str(destination), "rev-parse", "HEAD"], text=True).strip()
        if actual != revision:
            raise RuntimeError(f"{destination} 不是要求的版本；请使用新的运行时，不要覆盖旧目录。")
        if subprocess.check_output(["git", "-C", str(destination), "status", "--porcelain"], text=True).strip():
            raise RuntimeError(f"{destination} 有本地修改，请先检查。")
        return
    subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout", url, str(destination)], check=True)
    subprocess.run(["git", "-C", str(destination), "-c", "core.autocrlf=false", "checkout", "--detach", revision], check=True)

checkout_exact("https://github.com/Kaiyue2003/llm-design-bench.git", REPO, RELEASE_REVISION)
checkout_exact("https://github.com/namkoong-lab/data-recipes.git", UPSTREAM, UPSTREAM_REVISION)
ASSETS = REPO / "experiments/llmdm_forward_v1"
release = json.loads((ASSETS / "release.json").read_text())
print("Release checkout:", RELEASE_REVISION)
print("Frozen code commit:", release["code_commit"])
print("Data manifest:", release["data_manifest_id"])


## 1. 安装环境，保留 Colab 的 CUDA PyTorch

不要运行 data-recipes 的旧版 setup_env.sh，不安装它完整的优化框架依赖。此单元不训练、不调用 oracle。安装后如果 Colab 提示重启，请重启 Python 会话并重新运行准备单元。


In [ ]:
torch_before = importlib.metadata.version("torch")
constraints = Path("/content/llmdm-torch-constraint.txt")
constraints.write_text(f"torch=={torch_before}\n")
subprocess.run([sys.executable, "-m", "pip", "install", "-c", str(constraints), "-e", str(REPO)], check=True)
subprocess.run([sys.executable, "-m", "pip", "check"], check=True)
assert importlib.metadata.version("torch") == torch_before, "PyTorch 被更换，请检查环境"
os.environ["MPLCONFIGDIR"] = "/content/llmdm-mpl-cache"
import torch
print("Python:", sys.version)
print("PyTorch:", torch.__version__, "CUDA runtime:", torch.version.cuda)
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory GiB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))
subprocess.run(["free", "-h"], check=True)


## 2. 只读校验真实数据与冻结计划

应看到 logged=454、main visible=184、fixed-1B=26。这里不重新切分、不重新冻结预算，也不加载 oracle checkpoint。


In [ ]:
from llm_design_bench.evaluation.data_manifest import load_data_manifest
from llm_design_bench.evaluation.llmdm_protocol import load_method_plan

for relative, expected_hash in release["artifact_sha256"].items():
    actual_hash = hashlib.sha256((ASSETS / relative).read_bytes()).hexdigest()
    if actual_hash != expected_hash:
        raise RuntimeError(f"发布文件校验失败: {relative}")
bundle = load_data_manifest(ASSETS / "data", data_recipes_root=UPSTREAM)
plan = load_method_plan(ASSETS / "plan.json", bundle)
assert bundle.manifest_id == release["data_manifest_id"]
assert plan["plan_id"] == release["plan_id"]
assert len(bundle.reference_utility) == 454
assert len(bundle.utility) == 184
assert int((bundle.context[:, 0] == 1000).sum()) == 26
assert plan["package_source"]["git_dirty"] is False
print("校验通过：454 logged / 184 main visible / 26 fixed-1B")
print("可选方法:", ", ".join(entry["run_id"] for entry in plan["methods"]))
print("Frozen plan:", plan["plan_id"])


## 3. 挂载 Drive 与恢复备份

本地 /content 是实际运行位置；Drive 保存带 SHA256 校验的不可覆盖归档和启动记录。

这是你授权的 Drive 挂载操作，Google 会要求你选择账户。不要把未审查的 notebook 授予 Drive 权限。此 notebook 的结果路径限定为下面的 benchmark 目录。

恢复只会跳过已完成且校验通过的 seed。被中断的训练不会从中间参数继续；必须检查记录并填写基础设施中断原因，才能按原配置重跑。


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
sys.path.insert(0, str(REPO / "scripts"))
from colab_support import restore_latest, run_job

STATE = Path("/content/llmdm_run_state")
BACKUPS = Path("/content/drive/MyDrive/llm_design_bench/llmdm_forward_v1") / plan["plan_id"]
BACKUPS.mkdir(parents=True, exist_ok=True)
if not STATE.exists() or not any(STATE.iterdir()):
    if list((BACKUPS / "snapshots").glob("snapshot-*.tar.gz")):
        restored = restore_latest(BACKUPS, STATE)
        print("恢复归档:", restored)
    else:
        STATE.mkdir(parents=True, exist_ok=True)
        print("新的实验状态目录")
else:
    print("继续使用本运行时已有状态；不覆盖")
print("持久备份目录:", BACKUPS)


## 4. 选择一项任务

第一次可先选 best_logged / random_search / sobol 检查基线，再选择 offline_mlp 等需要训练的方法。RoMA、Tri-Mentoring、SPADE 等可能明显更慢，应逐个做完整 pilot。

- pilot 固定 seed=0，预算与正式实验完全相同。
- formal 只能用 38–45，并要求该方法、该设置的完整 pilot 成功。先人工确认资源、稳定性和文件，再允许 formal。
- 先完成主实验；fixed_1b 的 pilot/正式结果另存，并共享同一数据清单。
- 不根据 oracle 分数改变预算、选择 checkpoint 或挑选 seed。


In [ ]:
METHOD_ID = "offline_mlp"  # 选择 plan 中一个 run_id
SETTING = "multi_scale"  # multi_scale 或 fixed_1b
PHASE = "pilot"  # pilot 或 formal
FORMAL_SEED = 38  # 正式依次运行 38,39,40,41,42,43,44,45
NEURAL_DEVICE = "cuda"  # 在第一次 pilot 前决定；同一方法不要混用 CPU/GPU
RUN_JOB = False  # 看完配置后手动改为 True，才会训练/评估
CONFIRM_PILOT_REVIEWED = False  # formal 前确认已检查完整 pilot 的耗时、内存和数值稳定性
ALLOW_ENVIRONMENT_CHANGE = False  # 若环境变化，先检查可比性；不会自动替你放行
INFRASTRUCTURE_RETRY_REASON = ""  # 仅用于已核实的断线/VM回收等基础设施中断；算法失败不可冒充

CPU_METHODS = {"best_logged", "random_search", "sobol", "bdi", "bo_qei", "ga_on_gp"}
DEVICE = "cpu" if METHOD_ID in CPU_METHODS else NEURAL_DEVICE
SEED = 0 if PHASE == "pilot" else FORMAL_SEED
chosen = next((x for x in plan["methods"] if x["run_id"] == METHOD_ID), None)
if chosen is None or SETTING not in {"multi_scale", "fixed_1b"}:
    raise ValueError("请选择已冻结的方法和设置")
if PHASE not in {"pilot", "formal"} or (PHASE == "formal" and SEED not in range(38, 46)):
    raise ValueError("阶段或 seed 不合法")
if DEVICE not in {"cpu", "cuda"} or (DEVICE == "cuda" and not torch.cuda.is_available()):
    raise RuntimeError("需要 GPU 运行时，或在 pilot 前明确选用 CPU")
print(json.dumps({"run_id": METHOD_ID, "setting": SETTING, "phase": PHASE, "seed": SEED,
                  "device": DEVICE, "dtype": chosen["dtype"], "K": 128,
                  "kwargs": chosen["kwargs"]}, indent=2))


## 5. 执行这一项任务（有开关）

每次只启动一个子进程，期间定期备份，退出成功或失败后均备份。若 VM 被强制回收，最近一次未保存的训练状态仍可能丢失；下一次会通过启动记录要求人工确认，不能承诺零丢失。

同一个 Drive 备份目录不要开两个 Colab 会话同时写入。不要使用保持在线的脚本，也不要循环规避 Colab 的运行时限制。

备份是完整状态归档，旧归档不会自动删除。长批次前请检查 Drive 剩余空间，并监控备份目录大小；备份失败会停止当前任务并留下记录。


In [ ]:
if not RUN_JOB:
    print("未启动：确认以上配置后，把 RUN_JOB 改为 True，并重新运行选择单元和本单元。")
else:
    if PHASE == "formal" and not CONFIRM_PILOT_REVIEWED:
        raise RuntimeError("先人工检查完整 pilot，再设置 CONFIRM_PILOT_REVIEWED=True")
    tracked_packages = ["torch", "numpy", "pandas", "scipy", "scikit-learn", "plotly"]
    environment = {"device": DEVICE,
                   "python": platform.python_version(),
                   "cuda_runtime": torch.version.cuda,
                   "cudnn": torch.backends.cudnn.version(),
                   "gpu": torch.cuda.get_device_name(0) if DEVICE == "cuda" else None,
                   "packages": {name: importlib.metadata.version(name) for name in tracked_packages}}
    contract_path = STATE / f"environment-{METHOD_ID}.json"
    if contract_path.exists():
        before = json.loads(contract_path.read_text())
        if before["device"] != DEVICE:
            raise RuntimeError("同一方法 pilot 与正式运行设备类型不一致；请保持原设备类型。")
        if before != environment and not ALLOW_ENVIRONMENT_CHANGE:
            print("原环境:", before, "\n当前环境:", environment)
            raise RuntimeError("软件版本或GPU变化；请先审查，确认后才能显式放行。耗时不再直接可比。")
    else:
        contract_path.write_text(json.dumps(environment, indent=2))
    results_dir = STATE / PHASE
    command = [sys.executable, "-u", "-m", "llm_design_bench.llmdm_cli", "run",
               "--data-recipes-root", str(UPSTREAM), "--data-bundle", str(ASSETS / "data"),
               "--plan", str(ASSETS / "plan.json"), "--phase", PHASE,
               "--setting", SETTING, "--run-id", METHOD_ID, "--seed", str(SEED),
               "--device", DEVICE, "--oracle-device", "cpu", "--torch-threads", "1",
               "--results-dir", str(results_dir), "--resume"]
    if PHASE == "formal":
        command += ["--pilot-results", str(STATE / "pilot")]
    if INFRASTRUCTURE_RETRY_REASON.strip():
        command += ["--infrastructure-retry-reason", INFRASTRUCTURE_RETRY_REASON.strip()]
    identity = {"plan_id": plan["plan_id"], "run_id": METHOD_ID, "setting": SETTING,
                "phase": PHASE, "seed": SEED, "device": DEVICE, "oracle_device": "cpu"}
    job = run_job(command, STATE, BACKUPS, identity, snapshot_interval=60,
                  infrastructure_retry_reason=INFRASTRUCTURE_RETRY_REASON.strip() or None)
    print("任务结束，备份完成:", job)


## 6. 查看覆盖率和结果文件

成功一次不等于正式实验完成。每个方法、每种设置都应有 38–45 的 8 条成功记录；缺失或失败会保留，不完整结果不排名。这里只展示状态/运行成本，不依据 pilot oracle 分数选参数。


In [ ]:
import pandas as pd
summary_path = STATE / PHASE / "method_seed_summary.csv"
if summary_path.exists():
    summary = pd.read_csv(summary_path)
    columns = ["task_id", "run_id", "successful_runs", "failed_runs", "missing_runs",
               "rank_eligible", "method_seconds_mean", "peak_gpu_memory_bytes_mean"]
    display(summary[[c for c in columns if c in summary.columns]])
    print("逐候选、逐 seed 文件:", STATE / PHASE)
    print("Drive 备份:", BACKUPS)
else:
    print("还没有运行结果。")


## 接下来

完成所选方法的 seed-0 检查后，手动切换 PHASE=formal、CONFIRM_PILOT_REVIEWED=True，并逐次选择 38–45。切换方法时保持公共设置不变。不要直接修改 plan.json 或以较小预算替代完整 pilot。

[Colab 官方 FAQ](https://research.google.com/colaboratory/faq.html) · [项目实验协议](https://github.com/Kaiyue2003/llm-design-bench/blob/2a40f564dd33d117d322a57746d43fc174180cdf/docs/LLMDM_PROTOCOL.md)
